In [1]:
# ---------------------------------------------------------
# Cell 1: 匯入所有必要套件
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import warnings
import random
import os
import matplotlib.pyplot as plt
import optuna

from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, StackingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

random.seed(42)
np.random.seed(42)
warnings.filterwarnings('ignore')

/opt/anaconda3/envs/general/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---------------------------------------------------------
# Cell 2: 基礎資料清洗、極端值防禦與 Target Encoding
# ---------------------------------------------------------
# 1. 讀取原始資料
train_df = pd.read_csv('../data/boy or girl 2025 train_missingValue.csv')
test_df = pd.read_csv('../data/boy or girl 2025 test no ans_missingValue.csv')

# 2. 清理 phone_os 噪音值
def clean_os(val):
    val = str(val).lower()
    if 'apple' in val or 'ios' in val: return 'Apple'
    if 'android' in val: return 'Android'
    return 'Other'

train_df['phone_os'] = train_df['phone_os'].apply(clean_os)
test_df['phone_os'] = test_df['phone_os'].apply(clean_os)

# 3. 物理邊界強化 (嚴格過濾極端值，防溢位)
def apply_advanced_bounds(df):
    df_clean = df.copy()
    df_clean['yt'] = pd.to_numeric(df_clean['yt'], errors='coerce')
    df_clean['height'] = df_clean['height'].apply(lambda x: x if 100 <= x <= 250 else np.nan)
    df_clean['weight'] = df_clean['weight'].apply(lambda x: x if 30 <= x <= 200 else np.nan)
    df_clean['iq'] = df_clean['iq'].apply(lambda x: x if 50 <= x <= 200 else np.nan)
    df_clean['sleepiness'] = df_clean['sleepiness'].apply(lambda x: x if 0 <= x <= 24 else np.nan)
    df_clean['fb_friends'] = df_clean['fb_friends'].apply(lambda x: x if 0 <= x <= 5000 else np.nan)
    df_clean['yt'] = df_clean['yt'].apply(lambda x: x if 0 <= x <= 1e15 else np.nan)
    return df_clean

train_df = apply_advanced_bounds(train_df)
test_df = apply_advanced_bounds(test_df)

# 4. 保留類別特徵並進行 Target Encoding (平滑化對抗小樣本)
X = train_df.drop(columns=['id', 'gender', 'self_intro'])
y = train_df['gender']
X_test_submission = test_df.drop(columns=['id', 'gender', 'self_intro'])
test_ids = test_df['id']

le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)

print("執行 Target Encoding (Smoothing=20)...")
te = TargetEncoder(cols=['star_sign', 'phone_os'], smoothing=20)
X = te.fit_transform(X, y_encoded)
X_test_submission = te.transform(X_test_submission)

# 5. 建立缺失值指示器 (保留重要的缺失資訊)
for col in ['weight', 'height', 'yt']:
    X[f'{col}_is_missing'] = X[col].isnull().astype(int)
    X_test_submission[f'{col}_is_missing'] = X_test_submission[col].isnull().astype(int)

print(f"✅ Cell 2 處理完成！目前特徵總數: {X.shape[1]}")

執行 Target Encoding (Smoothing=20)...
✅ Cell 2 處理完成！目前特徵總數: 11


In [3]:
# ---------------------------------------------------------
# Cell 3: 升級版 NLP 語意萃取 (BGE-Large) 與 PCA 降維
# ---------------------------------------------------------
print("載入 BGE-Large 模型擷取文字語意 (首次執行需下載模型)...")
train_df['self_intro'] = train_df['self_intro'].fillna('')
test_df['self_intro'] = test_df['self_intro'].fillna('')

# 🚀 升級武器 1：使用 BGE-Large 擷取深層語意
embedder = SentenceTransformer('BAAI/bge-large-en-v1.5')
train_emb = embedder.encode(train_df['self_intro'].tolist(), show_progress_bar=True)
test_emb = embedder.encode(test_df['self_intro'].tolist(), show_progress_bar=True)

# 透過 PCA 將高維向量降至 5 維，防止維度災難
print("執行 5 維 Text Embedding PCA...")
pca = PCA(n_components=5, random_state=42)
train_pca = pca.fit_transform(train_emb)
test_pca = pca.transform(test_emb)

for i in range(5):
    X[f'intro_pca_{i}'] = train_pca[:, i]
    X_test_submission[f'intro_pca_{i}'] = test_pca[:, i]

print(f"✅ NLP 處理完成！加上 PCA 特徵後總數: {X.shape[1]}")

載入 BGE-Large 模型擷取文字語意 (首次執行需下載模型)...


Batches: 100%|██████████| 14/14 [00:03<00:00,  4.24it/s]

執行 5 維 Text Embedding PCA...
✅ NLP 處理完成！加上 PCA 特徵後總數: 16


In [4]:
# ---------------------------------------------------------
# Cell 4: 隨機森林進階補值與特徵工程
# ---------------------------------------------------------
print("開始使用 Random Forest 演算法插補缺失值...")

rf_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1), 
    random_state=42, 
    max_iter=10
)

# 補值並轉回 DataFrame
X_imputed = pd.DataFrame(rf_imputer.fit_transform(X), columns=X.columns)
X_test_imputed = pd.DataFrame(rf_imputer.transform(X_test_submission), columns=X_test_submission.columns)

print("開始建立進階衍生特徵 (BMI, 體型比例, Log Transform)...")

def create_advanced_features(df):
    df_new = df.copy()
    
    # 1. 建立 BMI 特徵
    df_new['height'] = df_new['height'].clip(lower=1) 
    df_new['BMI'] = df_new['weight'] / ((df_new['height'] / 100) ** 2)
    
    # 🚀 升級武器 2：加入線性的身高體重比 (來自 Willy)
    df_new['height_weight_ratio'] = df_new['height'] / df_new['weight']
    
    # 2. 對極端右偏的數值進行 Log1p 轉換
    df_new['yt_log'] = np.log1p(df_new['yt'].clip(lower=0))
    df_new['fb_friends_log'] = np.log1p(df_new['fb_friends'].clip(lower=0))
    
    # 移除原始尚未 Log 的欄位
    df_new = df_new.drop(columns=['yt', 'fb_friends'])
    return df_new

X_imputed_eng = create_advanced_features(X_imputed)
X_test_imputed_eng = create_advanced_features(X_test_imputed)

print(f"✅ 特徵工程完成！進入淘汰階段前總特徵數: {X_imputed_eng.shape[1]}")

開始使用 Random Forest 演算法插補缺失值...
開始建立進階衍生特徵 (BMI, 體型比例, Log Transform)...
✅ 特徵工程完成！進入淘汰階段前總特徵數: 18


In [5]:
# ---------------------------------------------------------
# Cell 5: 科學化特徵淘汰 (Permutation Importance)
# ---------------------------------------------------------
print("🚀 升級武器 3：執行 Permutation Importance 自動抓出噪聲特徵...")

# 使用 LightGBM (加上平衡權重) 作為測試模型
lgbm_pi = LGBMClassifier(class_weight='balanced', random_state=42, max_depth=4, verbose=-1)

# 用 5-Fold CV 計算，避免單一切分導致的誤判
cv_pi = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
perm_scores = []
features = X_imputed_eng.columns

print("計算重要性中，請稍候...")
for tr_idx, val_idx in cv_pi.split(X_imputed_eng, y_encoded):
    X_tr, y_tr = X_imputed_eng.iloc[tr_idx], y_encoded[tr_idx]
    X_val, y_val = X_imputed_eng.iloc[val_idx], y_encoded[val_idx]
    
    lgbm_pi.fit(X_tr, y_tr)
    
    # 用 ROC-AUC 作為淘汰標準
    result = permutation_importance(
        lgbm_pi, X_val, y_val,
        n_repeats=10, random_state=42, scoring='roc_auc'
    )
    perm_scores.append(result.importances_mean)

perm_importance_mean = np.mean(perm_scores, axis=0)

# 自動過濾 (只保留重要性大於 0 的特徵)
keep_mask = perm_importance_mean > 0
removed_features = [features[i] for i in range(len(features)) if not keep_mask[i]]
kept_features = [features[i] for i in range(len(features)) if keep_mask[i]]

print(f"\n🚫 成功移除噪聲特徵 (對 AUC 貢獻 <= 0): {removed_features}")
print(f"✅ 保留核心特徵 ({len(kept_features)} 個): {kept_features}")

# 產生純淨版 X_pruned
X_pruned = X_imputed_eng[kept_features].copy()
X_test_pruned = X_test_imputed_eng[kept_features].copy()

# 資料切割準備 (供後續 Optuna 與 CV 使用)
X_train, X_val, y_train, y_val = train_test_split(
    X_pruned, y_encoded, test_size=0.15, random_state=42, stratify=y_encoded
)
print(f"✅ 核心資料準備完成！維度: {X_pruned.shape}")

🚀 升級武器 3：執行 Permutation Importance 自動抓出噪聲特徵...
計算重要性中，請稍候...

🚫 成功移除噪聲特徵 (對 AUC 貢獻 <= 0): ['iq', 'yt_is_missing', 'intro_pca_0', 'intro_pca_1', 'yt_log', 'fb_friends_log']
✅ 保留核心特徵 (12 個): ['star_sign', 'phone_os', 'height', 'weight', 'sleepiness', 'weight_is_missing', 'height_is_missing', 'intro_pca_2', 'intro_pca_3', 'intro_pca_4', 'BMI', 'height_weight_ratio']
✅ 核心資料準備完成！維度: (423, 12)


In [6]:
# ---------------------------------------------------------
# Cell 6: Optuna 最佳化與終極 Stacking 驗證
# ---------------------------------------------------------
print("啟動 Optuna 自動尋找最強參數 (目標: 最大化 ROC-AUC)...")

ratio = (y_encoded == 0).sum() / (y_encoded == 1).sum()

def objective(trial):
    # XGBoost 參數 (保留 scale_pos_weight 抗失衡)
    xgb_params = {
        'scale_pos_weight': ratio, 'eval_metric': 'logloss', 'random_state': 42,
        'tree_method': 'hist', 'device': 'cuda', # 如果沒有 GPU 請將 device 改為 'cpu'
        'max_depth': trial.suggest_int('xgb_max_depth', 3, 7),
        'learning_rate': trial.suggest_float('xgb_lr', 0.01, 0.1, log=True),
        'n_estimators': trial.suggest_int('xgb_n', 100, 300),
        'subsample': trial.suggest_float('xgb_sub', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('xgb_col', 0.6, 1.0),
        'reg_lambda': trial.suggest_float('xgb_lambda', 1e-3, 10.0, log=True)
    }
    xgb_clf = XGBClassifier(**xgb_params)

    # LightGBM 參數 (保留 class_weight='balanced')
    lgbm_params = {
        'class_weight': 'balanced', 'random_state': 42, 'verbose': -1,
        'max_depth': trial.suggest_int('lgbm_max_depth', 3, 7),
        'num_leaves': trial.suggest_int('lgbm_leaves', 15, 63),
        'learning_rate': trial.suggest_float('lgbm_lr', 0.01, 0.1, log=True),
        'n_estimators': trial.suggest_int('lgbm_n', 100, 300)
    }
    lgbm_clf = LGBMClassifier(**lgbm_params)

    # Random Forest 參數 (保留 class_weight='balanced')
    rf_params = {
        'class_weight': 'balanced', 'random_state': 42,
        'max_depth': trial.suggest_int('rf_max_depth', 4, 10),
        'n_estimators': trial.suggest_int('rf_n', 100, 300),
        'min_samples_leaf': trial.suggest_int('rf_min_leaf', 1, 5)
    }
    rf_clf = RandomForestClassifier(**rf_params)

    # 組裝 Stacking 模型
    model = StackingClassifier(
        estimators=[('XGB', xgb_clf), ('LGBM', lgbm_clf), ('RF', rf_clf)],
        final_estimator=LogisticRegression(class_weight='balanced', random_state=42),
        cv=5
    )

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) 
    score = cross_validate(model, X_train, y_train, cv=skf, scoring='roc_auc')['test_score'].mean()
    return score

# 尋找最佳參數
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20) 

print("\n🏆 Optuna 搜尋完成！找到最佳參數組合：")
best_p = study.best_params

print("\n使用最佳參數建立最終的 Stacking 模型...")
final_xgb = XGBClassifier(
    scale_pos_weight=ratio, eval_metric='logloss', random_state=42, tree_method='hist', device='cuda',
    max_depth=best_p['xgb_max_depth'], learning_rate=best_p['xgb_lr'], n_estimators=best_p['xgb_n'],
    subsample=best_p['xgb_sub'], colsample_bytree=best_p['xgb_col'], reg_lambda=best_p['xgb_lambda']
)

final_lgbm = LGBMClassifier(
    class_weight='balanced', random_state=42, verbose=-1,
    max_depth=best_p['lgbm_max_depth'], num_leaves=best_p['lgbm_leaves'],
    learning_rate=best_p['lgbm_lr'], n_estimators=best_p['lgbm_n']
)

final_rf = RandomForestClassifier(
    class_weight='balanced', random_state=42,
    max_depth=best_p['rf_max_depth'], n_estimators=best_p['rf_n'], min_samples_leaf=best_p['rf_min_leaf']
)

stacking_model = StackingClassifier(
    estimators=[('XGB', final_xgb), ('LGBM', final_lgbm), ('RF', final_rf)],
    final_estimator=LogisticRegression(class_weight='balanced', random_state=42),
    cv=5
)

print("進行 Stratified 10-Fold 交叉驗證中 (評估終極版 Stacking 模型)...")
skf_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring_metrics = {'accuracy': 'accuracy', 'roc_auc': 'roc_auc', 'f1': 'f1', 'log_loss': 'neg_log_loss'}
cv_results = cross_validate(stacking_model, X_train, y_train, cv=skf_10, scoring=scoring_metrics)

print("\n📊 === Ultimate 終極版 Stacking 10-Fold CV 綜合評估報告 ===")
print(f"Accuracy (準確率) : {cv_results['test_accuracy'].mean():.4f} (std: {cv_results['test_accuracy'].std():.4f})")
print(f"ROC-AUC  (排序力) : {cv_results['test_roc_auc'].mean():.4f} (std: {cv_results['test_roc_auc'].std():.4f})")
print(f"F1-Score (平衡度) : {cv_results['test_f1'].mean():.4f} (std: {cv_results['test_f1'].std():.4f})")
print(f"Log Loss (誤差值) : {-cv_results['test_log_loss'].mean():.4f} (std: {cv_results['test_log_loss'].std():.4f})")

[I 2026-03-25 15:20:19,422] A new study created in memory with name: no-name-db19facb-579d-4bb5-897e-07d337e1b098


啟動 Optuna 自動尋找最強參數 (目標: 最大化 ROC-AUC)...


[I 2026-03-25 15:20:33,624] Trial 0 finished with value: 0.9239645526581419 and parameters: {'xgb_max_depth': 4, 'xgb_lr': 0.068311839148348, 'xgb_n': 266, 'xgb_sub': 0.7946826653403135, 'xgb_col': 0.773840361329669, 'xgb_lambda': 0.13991829322214586, 'lgbm_max_depth': 6, 'lgbm_leaves': 19, 'lgbm_lr': 0.05994049242671318, 'lgbm_n': 238, 'rf_max_depth': 9, 'rf_n': 184, 'rf_min_leaf': 3}. Best is trial 0 with value: 0.9239645526581419.
[I 2026-03-25 15:20:44,719] Trial 1 finished with value: 0.9222925120861788 and parameters: {'xgb_max_depth': 7, 'xgb_lr': 0.017213602341496353, 'xgb_n': 222, 'xgb_sub': 0.8266965539324576, 'xgb_col': 0.9063428836365036, 'xgb_lambda': 0.006249162258031546, 'lgbm_max_depth': 3, 'lgbm_leaves': 34, 'lgbm_lr': 0.09085870494453112, 'lgbm_n': 186, 'rf_max_depth': 9, 'rf_n': 194, 'rf_min_leaf': 3}. Best is trial 0 with value: 0.9239645526581419.
[I 2026-03-25 15:20:53,073] Trial 2 finished with value: 0.9262105590087494 and parameters: {'xgb_max_depth': 3, 'xgb_l


🏆 Optuna 搜尋完成！找到最佳參數組合：

使用最佳參數建立最終的 Stacking 模型...
進行 Stratified 10-Fold 交叉驗證中 (評估終極版 Stacking 模型)...

📊 === Ultimate 終極版 Stacking 10-Fold CV 綜合評估報告 ===
Accuracy (準確率) : 0.8775 (std: 0.0394)
ROC-AUC  (排序力) : 0.9281 (std: 0.0627)
F1-Score (平衡度) : 0.7673 (std: 0.0831)
Log Loss (誤差值) : 0.3531 (std: 0.0663)


In [7]:
# ---------------------------------------------------------
# Cell 7: 終極預測與存檔 (Ultimate 版)
# ---------------------------------------------------------
print(f"目前訓練集特徵數: {X_pruned.shape[1]}") 
print(f"目前測試集特徵數: {X_test_pruned.shape[1]}") 

# 用所有資料正式訓練終極版模型
print("正式訓練 Ultimate 終極版 Stacking 模型中...")
stacking_model.fit(X_pruned, y_encoded)
print("訓練完成！")

print("進行測試集預測中...")
test_predictions_encoded = stacking_model.predict(X_test_pruned) 

# 將 0, 1 轉回原始的性別標籤
test_predictions_original = le_y.inverse_transform(test_predictions_encoded)

submission_df = pd.DataFrame({
    'id': test_ids, 
    'gender': test_predictions_original
})

output_dir = '../results'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

file_path = os.path.join(output_dir, 'submission_ultimate.csv')
submission_df.to_csv(file_path, index=False)

print(f"🏆 終極版預測成功！檔案已儲存至: {file_path}")
print("前五筆預測結果：")
print(submission_df.head())

目前訓練集特徵數: 12
目前測試集特徵數: 12
正式訓練 Ultimate 終極版 Stacking 模型中...
訓練完成！
進行測試集預測中...
🏆 終極版預測成功！檔案已儲存至: ../results/submission_ultimate.csv
前五筆預測結果：
   id  gender
0   1       1
1   2       1
2   3       2
3   4       1
4   5       2
